# Assignment 2 - Model to Web Application
## Dataset 8: E-commerce Delivery Time Prediction

**Domain:** Operations and E-commerce
**Target variable:** `Delivery_Time_Hours` (continuous, hours)
**Primary model:** Multiple Linear Regression (scikit-learn `Pipeline`)

### Business problem
An e-commerce operations team promises a delivery window at the moment of checkout.
If the promise is too optimistic the company pays for expedited recovery, misses SLAs and
absorbs support contacts; if it is too conservative it loses conversions to faster
competitors. The team needs a model that turns *order and network attributes known at
dispatch time* - route distance, warehouse dwell time, courier saturation, road
congestion, parcel profile and service level - into an expected door-step time in hours,
plus an understanding of **which levers actually move that number**.

### Notebook contents
| Part | Content |
|---|---|
| **A** | Data quality, feature typing, and the full set of linear-regression assumption diagnostics with corrective action |
| **B** | 80/20 split, `ColumnTransformer` + `LinearRegression` pipeline, evaluation (R², Adj. R², MAE, MSE/RMSE), coefficient interpretation |
| **C** | Export of the fitted preprocessing + model pipeline to `model.pkl` with `joblib` |

> Run `python generate_data.py` first to produce `data.csv`, then run this notebook top to bottom.

## 0. Environment and imports

In [ ]:
# Core
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics / diagnostics
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from scipy import stats

# Modelling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
TARGET = "Delivery_Time_Hours"
ID_COLS = ["Order_ID"]

print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. Load the dataset

In [ ]:
df_raw = pd.read_csv("data.csv")
print("Shape:", df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="number").T.round(2)

---
# Part A - Data Preparation and Linear Regression Assumptions
---
## A1. Data quality audit

Before touching the model we check the five failure modes that matter for this file:
missing values, duplicate rows, impossible values, inconsistent categories, and
data-entry noise in the target.

In [ ]:
# --- Missing values -----------------------------------------------------
missing = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
})
print("MISSING VALUES")
print(missing[missing.missing_count > 0].sort_values("missing_count", ascending=False))

# --- Duplicates ---------------------------------------------------------
print("\nExact duplicate rows:", df_raw.duplicated().sum())
print("Duplicate Order_IDs  :", df_raw["Order_ID"].duplicated().sum())

In [ ]:
# --- Inconsistent categories -------------------------------------------
for col in ["Product_Category", "Shipping_Mode"]:
    print(f"\n{col} - raw levels ({df_raw[col].nunique()}):")
    print(df_raw[col].value_counts(dropna=False).to_string())

In [ ]:
# --- Impossible / out-of-range values ----------------------------------
# NOTE: NaN is "missing", not "impossible" - each rule requires a present value
checks = {
    "Order_Value <= 0":                lambda s: s <= 0,
    "Package_Weight_Kg <= 0":          lambda s: s <= 0,
    "Warehouse_Distance_Km <= 0":      lambda s: s <= 0,
    "Items_in_Order < 1":              lambda s: s < 1,
    "Warehouse_Processing_Hours < 0":  lambda s: s < 0,
    "Courier_Load_Index outside 1-10": lambda s: ~s.between(1, 10),
    "Traffic_Index outside 1-10":      lambda s: ~s.between(1, 10),
    "Delivery_Time_Hours <= 0":        lambda s: s <= 0,
}
col_of = {k: k.split(" ")[0] for k in checks}
print("IMPOSSIBLE-VALUE CHECKS")
for label, rule in checks.items():
    col = col_of[label]
    mask = df_raw[col].notna() & rule(df_raw[col])
    print(f"  {label:34s} -> {int(mask.sum())} rows")

**Diagnosis.** The extract is a realistic operational feed, not a clean file:

* **Missing values** in `Traffic_Index` (~2.0%), `Order_Value` (~1.7%), `Package_Weight_Kg`
  (~1.0%) and `Warehouse_Processing_Hours` (~0.7%) - typical of telemetry/integration gaps.
* **15 exact duplicate rows**, i.e. the same order ingested twice.
* **Inconsistent category labels** from multi-source ingestion: `electronics`, `APPAREL `,
  ` Home_Kitchen`, `groceries`, `standard`, `EXPRESS`, `Same Day`. Left untreated,
  one-hot encoding would create *two separate columns for the same business category* and
  silently split the effect.
* **Impossible values**: negative weight and distance, zero-item orders, `Traffic_Index = 99`
  on a 1-10 scale, and negative delivery times.

**Action taken (next cell).**
1. Normalise category spelling/whitespace/case before encoding.
2. Drop exact duplicates (they inflate n and bias standard errors downwards).
3. Drop the identifier `Order_ID` - it is a unique key, has no predictive content, and
   would leak row identity into the model.
4. Convert impossible values to `NaN` rather than clipping - a negative weight is not a
   small weight, it is an unknown weight.
5. Drop rows with a missing/invalid **target** (the target cannot be imputed without
   inventing the label), and median-impute missing **predictors** (robust to the skew in
   `Order_Value`).

In [ ]:
df = df_raw.copy()

# 1. Normalise categorical labels -------------------------------------------
def normalise_category(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.strip()
             .str.replace(r"[\s\-]+", "_", regex=True)   # "Same Day"/"Same-Day" -> "Same_Day"
             .str.lower()
             .str.replace("_", " ", regex=False)
             .str.title()
             .str.replace(" ", "_", regex=False))

for col in ["Product_Category", "Shipping_Mode"]:
    df[col] = normalise_category(df[col])
    print(f"{col}: {sorted(df[col].unique())}")

# 2. Duplicates -------------------------------------------------------------
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nDropped {before - len(df)} duplicate rows -> {len(df)} rows")

# 3. Identifier column ------------------------------------------------------
df = df.drop(columns=ID_COLS)
print("Dropped identifier column(s):", ID_COLS)

In [ ]:
# 4. Impossible values -> NaN ----------------------------------------------
invalid_rules = {
    "Order_Value":                 lambda s: s <= 0,
    "Package_Weight_Kg":           lambda s: s <= 0,
    "Warehouse_Distance_Km":       lambda s: s <= 0,
    "Items_in_Order":              lambda s: s < 1,
    "Warehouse_Processing_Hours":  lambda s: s < 0,
    "Courier_Load_Index":          lambda s: ~s.between(1, 10),
    "Traffic_Index":               lambda s: ~s.between(1, 10),
    TARGET:                        lambda s: s <= 0,
}
for col, rule in invalid_rules.items():
    mask = df[col].notna() & rule(df[col])   # NaN stays "missing", not "impossible"
    if mask.sum():
        print(f"{col:28s}: {int(mask.sum())} impossible value(s) -> NaN")
    df.loc[mask, col] = np.nan

# 5a. Missing target -> drop the row ---------------------------------------
before = len(df)
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print(f"\nDropped {before - len(df)} rows with an unusable target")

# 5b. Missing predictors -> median imputation ------------------------------
NUM_FEATURES = ["Order_Value", "Package_Weight_Kg", "Warehouse_Distance_Km",
                "Items_in_Order", "Warehouse_Processing_Hours",
                "Courier_Load_Index", "Traffic_Index"]
CAT_FEATURES = ["Product_Category", "Shipping_Mode"]

impute_values = df[NUM_FEATURES].median()
print("\nMedian imputation values:")
print(impute_values.round(2).to_string())
df[NUM_FEATURES] = df[NUM_FEATURES].fillna(impute_values)
df["Items_in_Order"] = df["Items_in_Order"].round().astype(int)

print("\nRemaining missing values:", int(df.isna().sum().sum()))
print("Clean shape:", df.shape)

## A2. Target and feature understanding

| Role | Columns |
|---|---|
| **Dependent variable (y)** | `Delivery_Time_Hours` - continuous, measured in hours |
| **Numerical predictors** | `Order_Value`, `Package_Weight_Kg`, `Warehouse_Distance_Km`, `Items_in_Order`, `Warehouse_Processing_Hours`, `Courier_Load_Index`, `Traffic_Index` |
| **Categorical predictors (nominal)** | `Product_Category` (6 levels), `Shipping_Mode` (3 levels) |
| **Identifier - excluded** | `Order_ID` |

`Courier_Load_Index` and `Traffic_Index` are bounded 1-10 indices. They are treated as
continuous because the spacing between points is meaningful and the scale has more than a
handful of distinct values. `Items_in_Order` is a count and is also treated as continuous -
a per-item marginal effect is exactly the quantity operations cares about.

In [ ]:
print("Target summary (hours):")
print(df[TARGET].describe().round(2).to_string())
print("\nSkewness:", round(stats.skew(df[TARGET]), 3))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
sns.histplot(df[TARGET], bins=40, kde=True, ax=ax[0])
ax[0].set_title("Distribution of Delivery_Time_Hours")
sns.boxplot(x=df[TARGET], ax=ax[1])
ax[1].set_title("Boxplot - long right tail visible")
plt.tight_layout(); plt.show()

In [ ]:
# Categorical structure vs target
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
order = df.groupby("Product_Category")[TARGET].median().sort_values().index
sns.boxplot(data=df, x="Product_Category", y=TARGET, order=order, ax=ax[0])
ax[0].tick_params(axis="x", rotation=30); ax[0].set_title("Delivery time by product category")
sns.boxplot(data=df, x="Shipping_Mode", y=TARGET,
            order=["Same_Day", "Express", "Standard"], ax=ax[1])
ax[1].set_title("Delivery time by shipping mode")
plt.tight_layout(); plt.show()

print(df.groupby("Shipping_Mode")[[TARGET, "Warehouse_Distance_Km"]].agg(["mean", "count"]).round(2))

**Read.** `Shipping_Mode` separates the target strongly, but note it is also *confounded
with distance*: `Same_Day` is only offered on short intra-city lanes and `Standard` carries
all the long national lanes. The regression handles this correctly by estimating the mode
effect *holding distance constant* - which is precisely the number an operations manager
needs ("what does upgrading this lane to Express buy me?").

## A3. Linearity

Two views: raw scatter of each continuous predictor against the target with a LOWESS-style
fit, and (after fitting) partial-residual plots, which isolate one predictor's relationship
while controlling for the others.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), NUM_FEATURES):
    sns.regplot(data=df, x=col, y=TARGET, ax=ax, scatter_kws=dict(s=8, alpha=0.35),
                line_kws=dict(color="crimson"), lowess=True)
    ax.set_title(f"{col} (r = {df[col].corr(df[TARGET]):+.2f})", fontsize=10)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
print("Pearson correlation with the target:")
print(df[NUM_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET)
        .sort_values(ascending=False).round(3).to_string())

**Diagnosis.** `Warehouse_Distance_Km` is the dominant driver (r ≈ +0.69) and its LOWESS
curve is essentially a straight line across the full 2-1500 km range. `Warehouse_Processing_Hours`,
`Traffic_Index` and `Courier_Load_Index` show clear positive linear trends; their raw
marginal correlations look modest only because distance and shipping mode dominate the
unconditional variance. `Order_Value` is right-skewed but its relationship with the target
is flat, not curved.

**Action.** No transformation of predictors is required at this stage - no fan, hook, or
saturation pattern is present. This is re-verified with partial-residual plots in **A4**,
which are the correct test of linearity for a *multiple* regression. A log transform of
`Order_Value` was considered for skewness alone and rejected: skewness of a predictor is not
a regression assumption, and the transform did not improve fit.

## A4. Baseline OLS fit (diagnostic engine for A4-A9)

`statsmodels` OLS is used for the assumption diagnostics because it exposes residuals,
leverage, influence and inference. The deliverable model in Part B is the scikit-learn
pipeline; both fit the same specification.

In [ ]:
# Nominal categories -> dummy variables, first level dropped to avoid the dummy trap
X_design = pd.get_dummies(df[NUM_FEATURES + CAT_FEATURES],
                          columns=CAT_FEATURES, drop_first=True, dtype=float)
y = df[TARGET].astype(float)

print("Design matrix:", X_design.shape)
print("Reference (dropped) levels: Product_Category = Apparel, Shipping_Mode = Express")
print(list(X_design.columns))

ols_base = sm.OLS(y, sm.add_constant(X_design)).fit()
print(ols_base.summary())

In [ ]:
# Partial-residual (component + residual) plots:
# linearity of one predictor while controlling for all the others
resid_base = ols_base.resid
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), NUM_FEATURES):
    component = ols_base.params[col] * X_design[col]
    partial = component + resid_base
    ax.scatter(X_design[col], partial, s=8, alpha=.30)
    sns.regplot(x=X_design[col], y=partial, lowess=True, scatter=False,
                line_kws=dict(color="crimson"), ax=ax)
    ax.plot(np.sort(X_design[col]), np.sort(component) if ols_base.params[col] > 0
            else np.sort(component)[::-1], color="black", lw=1.2, ls="--")
    ax.set_xlabel(col); ax.set_ylabel("partial residual (h)")
    ax.set_title(f"{col}  (slope = {ols_base.params[col]:+.4f})", fontsize=9)
axes.ravel()[-1].axis("off")
plt.suptitle("Partial-residual plots: dashed = fitted linear component, red = LOWESS", y=1.01)
plt.tight_layout(); plt.show()

**Interpretation.** Each partial-residual cloud is centred on its fitted straight line with
no systematic curvature, so **linearity in the parameters is satisfied** for every continuous
predictor. Note the baseline R² (~0.67) and the extreme residual skew in the OLS summary -
both are symptoms of the influential observations isolated in **A8**, not of a wrong
functional form.

## A5. Multicollinearity - correlation matrix and VIF

In [ ]:
corr = df[NUM_FEATURES].corr()
plt.figure(figsize=(7.5, 5.5))
sns.heatmap(corr, annot=True, fmt="+.2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, linewidths=.5, cbar_kws={"shrink": .8})
plt.title("Correlation among numerical predictors"); plt.tight_layout(); plt.show()

high = (corr.where(~np.eye(len(corr), dtype=bool)).abs().stack()
            .sort_values(ascending=False).head(5))
print("Strongest predictor-predictor correlations:")
print(high.round(3).to_string())

In [ ]:
# VIF on the numerical predictors (constant added, as VIF is defined on the design matrix)
vif_X = sm.add_constant(df[NUM_FEATURES].astype(float))
vif = pd.DataFrame({
    "feature": vif_X.columns,
    "VIF": [variance_inflation_factor(vif_X.values, i) for i in range(vif_X.shape[1])],
}).query("feature != 'const'").sort_values("VIF", ascending=False)
vif["verdict"] = pd.cut(vif.VIF, [0, 5, 10, np.inf],
                        labels=["OK (<5)", "Monitor (5-10)", "Severe (>10)"])
print(vif.round(3).to_string(index=False))

**Diagnosis.** The strongest pairwise correlation is `Courier_Load_Index` vs `Traffic_Index`
(≈ +0.34, congested days also load couriers) followed by the basket-size cluster
(`Items_in_Order` - `Package_Weight_Kg` - `Order_Value`). Every VIF is ≈ 1.0-1.3, far below
the conventional thresholds of 5 and 10.

**Action.** **No remediation required.** Nothing is dropped or combined: all seven numerical
predictors carry largely independent information, coefficient standard errors are not
inflated, and each coefficient remains individually interpretable - which matters because
the business use of this model is lever identification, not just prediction.

## A6. Independence of errors - Durbin-Watson

In [ ]:
dw = durbin_watson(ols_base.resid)
print(f"Durbin-Watson statistic: {dw:.4f}")
print("Rule of thumb: ~2.0 = no autocorrelation | <1.5 positive | >2.5 negative")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(ols_base.resid.values[:200], lw=.9)
ax[0].axhline(0, color="crimson", ls="--"); ax[0].set_title("Residuals in row order (first 200)")
ax[1].scatter(ols_base.resid.values[:-1], ols_base.resid.values[1:], s=8, alpha=.4)
ax[1].set_xlabel("residual $e_t$"); ax[1].set_ylabel("residual $e_{t+1}$")
ax[1].set_title("Lag-1 residual scatter")
plt.tight_layout(); plt.show()

**Diagnosis.** DW ≈ 2.01, effectively the no-autocorrelation ideal, and the lag-1 scatter is
a structureless cloud.

**Is independence a reasonable assumption here?** Yes, with one caveat worth stating.
Each row is an independent order and the file carries no timestamp, route or courier key, so
rows are exchangeable and Durbin-Watson (a *sequence* test) is only weakly informative.
In a production feed the real risk is **clustering, not serial correlation**: orders sharing
a warehouse-day or a courier route would have correlated errors. The mitigation in that
setting is clustered or robust standard errors, not a change to the point estimates.
**No action needed for this dataset.**

## A7. Homoscedasticity - residuals vs fitted and Breusch-Pagan

In [ ]:
fitted, resid = ols_base.fittedvalues, ols_base.resid
std_resid = ols_base.get_influence().resid_studentized_internal

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(fitted, resid, s=10, alpha=.4)
ax[0].axhline(0, color="crimson", ls="--")
ax[0].set_xlabel("Fitted values (hours)"); ax[0].set_ylabel("Residuals")
ax[0].set_title("Residuals vs Fitted")
ax[1].scatter(fitted, np.sqrt(np.abs(std_resid)), s=10, alpha=.4)
sns.regplot(x=fitted, y=np.sqrt(np.abs(std_resid)), lowess=True, scatter=False,
            line_kws=dict(color="crimson"), ax=ax[1])
ax[1].set_xlabel("Fitted values"); ax[1].set_ylabel(r"$\sqrt{|standardised\ residual|}$")
ax[1].set_title("Scale-Location")
plt.tight_layout(); plt.show()

In [ ]:
lm, lm_p, f_stat, f_p = het_breuschpagan(resid, ols_base.model.exog)
print("Breusch-Pagan test (H0: homoscedastic errors)")
print(f"  LM statistic = {lm:.3f}   p-value = {lm_p:.6f}")
print(f"  F statistic  = {f_stat:.3f}   p-value = {f_p:.6f}")
print("  ->", "Reject H0: heteroscedasticity" if lm_p < .05 else "Fail to reject H0")

**Diagnosis.** On this baseline fit the Breusch-Pagan test does **not** reject homoscedasticity
(p ≈ 0.63) - but that result should not be trusted yet. A handful of very large residuals
(see A8) dominate the squared-residual regression and mask the real pattern. The test is
re-run after the influence treatment in **A9**, where a genuine, mild variance-scaling
effect does appear. Treatment is deferred to A9 so that diagnosis and action are not based
on a contaminated fit.

## A8. Normality of residuals

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(resid, bins=50, kde=True, ax=ax[0])
ax[0].set_title("Histogram of residuals"); ax[0].set_xlabel("Residual (hours)")
sm.qqplot(resid, line="45", fit=True, ax=ax[1])
ax[1].set_title("Normal Q-Q plot of residuals")
plt.tight_layout(); plt.show()

jb_stat, jb_p, skew_, kurt_ = jarque_bera(resid)
print(f"Residual skewness = {skew_:.3f} | kurtosis = {kurt_:.3f}")
print(f"Jarque-Bera: stat = {jb_stat:.2f}, p = {jb_p:.3g}")
print(f"Shapiro-Wilk: stat = {stats.shapiro(resid)[0]:.4f}, p = {stats.shapiro(resid)[1]:.3g}")

**Diagnosis.** Severe violation *before* treatment: residual skewness ≈ 13, kurtosis ≈ 185,
Jarque-Bera p ≈ 0. The Q-Q plot is straight through the middle and then flies off at the
top - the classic signature of **a few extreme cases, not a globally mis-specified error
distribution**. Note that the assumption applies to the *residuals*, not to the input
features, so the skew in `Order_Value` is irrelevant here.

**Action.** Do not transform yet. Identify the offending observations first (**A8b**),
decide on them on business grounds, then re-test normality (**A9**).

## A8b. Outliers, leverage and influential observations

In [ ]:
infl = OLSInfluence(ols_base)
diag = pd.DataFrame({
    "std_resid": infl.resid_studentized_internal,
    "leverage": infl.hat_matrix_diag,
    "cooks_d": infl.cooks_distance[0],
})
n, p = ols_base.nobs, ols_base.df_model + 1
cook_cut, lev_cut = 4 / n, 3 * p / n
print(f"Thresholds: |std. residual| > 3 | leverage > 3p/n = {lev_cut:.4f} "
      f"| Cook's D > 4/n = {cook_cut:.5f} (and the strict D > 1)")
print(f"\nCount |std resid| > 3 : {(diag.std_resid.abs() > 3).sum()}")
print(f"Count high leverage   : {(diag.leverage > lev_cut).sum()}")
print(f"Count Cook's D > 4/n  : {(diag.cooks_d > cook_cut).sum()}")
print(f"Count Cook's D > 1    : {(diag.cooks_d > 1).sum()}  (max D = {diag.cooks_d.max():.3f})")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].stem(np.arange(len(diag)), diag.cooks_d, markerfmt=",", basefmt=" ")
ax[0].axhline(cook_cut, color="crimson", ls="--", label="4/n")
ax[0].set_title("Cook's distance"); ax[0].set_xlabel("observation"); ax[0].legend()

ax[1].scatter(diag.leverage, diag.std_resid, s=12, alpha=.5)
ax[1].axhline(3, color="crimson", ls="--"); ax[1].axhline(-3, color="crimson", ls="--")
ax[1].axvline(lev_cut, color="darkorange", ls="--")
ax[1].set_xlabel("Leverage"); ax[1].set_ylabel("Standardised residual")
ax[1].set_title("Influence plot (leverage vs std. residual)")

sns.histplot(df[TARGET], bins=60, ax=ax[2])
ax[2].axvline(df[TARGET].quantile(.75) + 3 * stats.iqr(df[TARGET]), color="crimson",
              ls="--", label="Q3 + 3*IQR")
ax[2].set_title("Target distribution with far-outlier fence"); ax[2].legend()
plt.tight_layout(); plt.show()

flagged = diag[diag.std_resid.abs() > 4].index
print("Observations with |std. residual| > 4:")
print(df.loc[flagged, ["Shipping_Mode", "Warehouse_Distance_Km",
                       "Warehouse_Processing_Hours", "Traffic_Index", TARGET]]
        .join(diag.loc[flagged].round(3)).to_string())

**Diagnosis and business investigation.** Six observations have standardised residuals
beyond ±4 (all positive, 66-116 hours) and by far the largest Cook's distances. Their
*predictors are entirely ordinary* - short-to-moderate lanes, normal dwell time, normal
congestion - so the model cannot explain them from anything recorded in the file. No point
exceeds the strict Cook's D > 1 threshold, but each is 5-20x the median influence.
Operationally these are the recognisable failure tail: customs/compliance holds, failed
first delivery attempts, address disputes, weather embargoes. They are *real events* but
they are **driven by variables the dataset does not contain**, so they cannot inform the
coefficients - they can only distort them.

The 14 high-leverage points are a different story: they are long-haul 1,200-1,500 km lanes.
They are unusual in the predictor space but sit on the fitted surface (small residuals), and
they are legitimate, repeating business. **They are retained** - removing them would shrink
the range over which the model is valid.

**Action.**
* **Remove** the 6 unexplainable extreme cases from the training data, and document it: the
  model predicts *normal-course delivery time*, and these exceptions belong to a separate
  exception-management process (a classifier for "will this order get stuck?"), not to a
  conditional-mean regression.
* **Retain** all high-leverage long-haul rows.
* No capping/winsorising of the target: capping would silently manufacture a fake value for
  a real event.

In [ ]:
keep = diag.std_resid.abs() <= 4
df_model = df.loc[keep.values].reset_index(drop=True)
print(f"Removed {int((~keep).sum())} influential exception rows -> {len(df_model)} rows retained "
      f"({len(df_model)/len(df):.2%} of the clean data)")
print(f"\nTarget range now: {df_model[TARGET].min():.2f} - {df_model[TARGET].max():.2f} hours")

## A9. Re-check assumptions after correction

In [ ]:
X_design2 = pd.get_dummies(df_model[NUM_FEATURES + CAT_FEATURES],
                           columns=CAT_FEATURES, drop_first=True, dtype=float)
y2 = df_model[TARGET].astype(float)
ols_fix = sm.OLS(y2, sm.add_constant(X_design2)).fit()

resid2, fitted2 = ols_fix.resid, ols_fix.fittedvalues
dw2 = durbin_watson(resid2)
lm2, lm2_p, _, _ = het_breuschpagan(resid2, ols_fix.model.exog)
jb2, jb2_p, skew2, kurt2 = jarque_bera(resid2)

summary = pd.DataFrame({
    "diagnostic": ["R-squared", "Durbin-Watson", "Breusch-Pagan p", "Jarque-Bera p",
                   "Residual skew", "Residual kurtosis", "Max Cook's D"],
    "before": [ols_base.rsquared, dw, lm_p, jb_p, skew_, kurt_, diag.cooks_d.max()],
    "after":  [ols_fix.rsquared, dw2, lm2_p, jb2_p, skew2, kurt2,
               OLSInfluence(ols_fix).cooks_distance[0].max()],
})
print(summary.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].scatter(fitted2, resid2, s=10, alpha=.4); ax[0].axhline(0, color="crimson", ls="--")
ax[0].set_title(f"Residuals vs Fitted (BP p = {lm2_p:.4f})")
ax[0].set_xlabel("Fitted (hours)"); ax[0].set_ylabel("Residual")
sm.qqplot(resid2, line="45", fit=True, ax=ax[1]); ax[1].set_title(f"Q-Q plot (JB p = {jb2_p:.3f})")
sns.histplot(resid2, bins=45, kde=True, ax=ax[2]); ax[2].set_title("Residual histogram")
plt.tight_layout(); plt.show()

**Result of the correction.**

| Assumption | Before | After | Verdict |
|---|---|---|---|
| Linearity | OK | OK | Satisfied (partial-residual plots straight) |
| No multicollinearity | VIF ≤ 1.3 | VIF ≤ 1.3 | Satisfied |
| Independence | DW 2.01 | DW 2.05 | Satisfied |
| Normality of residuals | JB p ≈ 0, skew ≈ 13 | **JB p ≈ 0.87, skew ≈ 0.0** | **Fixed** |
| Homoscedasticity | BP p 0.63 (masked) | BP p < 0.001 | **Violated - mild, treated below** |
| Influence | 6 points with 5-20x median Cook's D | max D ≈ 0.04 | **Fixed** |

R² jumps from 0.67 to **0.967** purely from removing six unexplainable exception rows - a
clean demonstration of how much a handful of influential points can distort OLS.

Normality is now essentially textbook. The remaining issue is heteroscedasticity: with the
outlier mask removed, Breusch-Pagan now detects the genuine (and expected) pattern that
absolute timing error grows with the length of the journey - a 40-hour cross-country
delivery is intrinsically more variable than a 6-hour intra-city hop.

In [ ]:
# Option 1: log-transform the target
ols_log = sm.OLS(np.log(y2), sm.add_constant(X_design2)).fit()
lm_log, lm_log_p, _, _ = het_breuschpagan(ols_log.resid, ols_log.model.exog)
print(f"log(target) model : R2 = {ols_log.rsquared:.4f}, Breusch-Pagan p = {lm_log_p:.6f}")
print(f"level model       : R2 = {ols_fix.rsquared:.4f}, Breusch-Pagan p = {lm2_p:.6f}")

# Option 2: keep the level model, use heteroscedasticity-robust (HC3) standard errors
ols_robust = ols_fix.get_robustcov_results(cov_type="HC3")
comparison = pd.DataFrame({
    "coefficient": ols_fix.params.round(4),
    "SE_classical": ols_fix.bse.round(4),
    "SE_robust_HC3": np.round(ols_robust.bse, 4),
    "p_robust": np.round(ols_robust.pvalues, 4),
})
print("\nLevel model with HC3 robust inference:")
print(comparison.to_string())

**Action on heteroscedasticity - and the reasoning.**

Two candidate treatments were tested:

1. **log-transform the target.** It makes the problem *worse* (Breusch-Pagan p still < 0.001,
   R² drops to ≈ 0.92) because the error scale here grows roughly *additively* with duration,
   not multiplicatively. It also converts every coefficient into a percentage effect, which
   is a worse answer for a dispatcher who thinks in hours. **Rejected.**
2. **Keep the level model and use HC3 heteroscedasticity-robust standard errors.** OLS
   coefficients stay unbiased and directly interpretable in hours under heteroscedasticity -
   only their standard errors are affected, and HC3 repairs exactly that. **Adopted.**

The robust table above confirms the substantive conclusions are unchanged: distance,
processing hours, traffic, courier load, item count, parcel weight, shipping mode and the
Electronics / Home_Kitchen / Groceries category effects all stay significant at the 1% level,
while `Order_Value` is only marginal (robust p ≈ 0.06) and the `Beauty` / `Books` category
effects remain indistinguishable from zero.

**Remaining limitation to declare:** a single prediction interval of constant width is not
appropriate for this model. The point prediction is unbiased, but the app should communicate
a *wider* uncertainty band on long lanes than on short ones. The Streamlit app therefore
presents a distance-scaled confidence band rather than one fixed ± figure.

---
# Part B - Build and Evaluate the Machine Learning Model
---
## B1. Train/test split (80/20)

In [ ]:
FEATURES = NUM_FEATURES + CAT_FEATURES
X = df_model[FEATURES].copy()
y = df_model[TARGET].astype(float).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)

print(f"Split: 80/20, random_state={RANDOM_STATE}")
print(f"  Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"  Train target mean {y_train.mean():.2f} h | Test target mean {y_test.mean():.2f} h")

## B2. Reproducible preprocessing + model pipeline

All preprocessing lives **inside** the pipeline, so the identical transformations are applied
at prediction time in the web app. This removes the single most common cause of
train/serve skew.

* `StandardScaler` on the 7 numerical predictors - puts coefficients on a comparable
  importance scale.
* `OneHotEncoder(drop="first")` on the 2 nominal predictors - avoids the dummy-variable trap
  and, critically, avoids imposing a fake ordinal ranking on categories that have no natural
  order. `handle_unknown="ignore"` keeps the app from crashing on an unseen category.
* `LinearRegression` as the primary model.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT_FEATURES),
    ],
    remainder="drop",
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

pipeline.fit(X_train, y_train)
print(pipeline)

## B3. Evaluation - R², Adjusted R², MAE, MSE, RMSE

In [ ]:
def evaluate(model, X_, y_, label):
    pred = model.predict(X_)
    n_, k_ = len(y_), model[:-1].transform(X_).shape[1]
    r2 = r2_score(y_, pred)
    mse = mean_squared_error(y_, pred)
    return {
        "set": label, "n": n_,
        "R2": r2,
        "Adj_R2": 1 - (1 - r2) * (n_ - 1) / (n_ - k_ - 1),
        "MAE": mean_absolute_error(y_, pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAPE_%": np.mean(np.abs((y_ - pred) / y_)) * 100,
    }

results = pd.DataFrame([evaluate(pipeline, X_train, y_train, "Train"),
                        evaluate(pipeline, X_test, y_test, "Test")])
print(results.round(4).to_string(index=False))

cv = cross_val_score(pipeline, X, y, cv=5, scoring="r2")
print(f"\n5-fold CV R²: {cv.round(4)}  ->  mean {cv.mean():.4f} (+/- {cv.std():.4f})")

**Read.** Test R² ≈ 0.968 with Adjusted R² ≈ 0.966, MAE ≈ 1.2 hours and RMSE ≈ 1.5 hours on a
target averaging ~19 hours - roughly a **6% average error**. Train and test metrics are
within 0.001 of each other and 5-fold CV is stable, so there is no overfitting: expected for
a 16-column linear model on ~1,190 rows.

RMSE (1.52) sits only slightly above MAE (1.22), confirming there is no longer a small set of
huge errors dragging the loss - consistent with the influence treatment in A8b.

## B4. Model coefficients and business interpretation

In [ ]:
feat_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
coefs = pipeline.named_steps["model"].coef_
train_sd = X_train[NUM_FEATURES].std()

rows = []
for name, c in zip(feat_names, coefs):
    block, raw = name.split("__", 1)
    if block == "num":
        rows.append({"feature": raw, "type": "numeric",
                     "coef_scaled(h per 1 SD)": c,
                     "coef_per_unit(h)": c / train_sd[raw],
                     "unit": {"Order_Value": "per currency unit",
                              "Package_Weight_Kg": "per kg",
                              "Warehouse_Distance_Km": "per km",
                              "Items_in_Order": "per item",
                              "Warehouse_Processing_Hours": "per hour",
                              "Courier_Load_Index": "per index point",
                              "Traffic_Index": "per index point"}[raw]})
    else:
        rows.append({"feature": raw, "type": "category (vs reference)",
                     "coef_scaled(h per 1 SD)": c, "coef_per_unit(h)": c,
                     "unit": "hours vs reference level"})

coef_tbl = (pd.DataFrame(rows)
            .assign(abs_impact=lambda d: d["coef_scaled(h per 1 SD)"].abs())
            .sort_values("abs_impact", ascending=False).drop(columns="abs_impact"))
print(f"Intercept (hours at mean numeric values, Apparel + Express): "
      f"{pipeline.named_steps['model'].intercept_:.3f}\n")
print(coef_tbl.round(5).to_string(index=False))

In [ ]:
plt.figure(figsize=(8.5, 5))
plot_df = coef_tbl.sort_values("coef_scaled(h per 1 SD)")
colors = ["#c0392b" if v > 0 else "#2471a3" for v in plot_df["coef_scaled(h per 1 SD)"]]
plt.barh(plot_df.feature, plot_df["coef_scaled(h per 1 SD)"], color=colors)
plt.axvline(0, color="black", lw=.8)
plt.xlabel("Change in delivery hours (per 1 SD, or vs reference category)")
plt.title("Standardised driver importance - Delivery_Time_Hours")
plt.tight_layout(); plt.show()

### What this means for an operations manager

Standardised coefficients answer *"which lever is worth pulling?"*; per-unit coefficients
answer *"what does one more unit cost me?"*

**Tier 1 - the levers that actually move delivery time**

1. **`Warehouse_Distance_Km` : +5.54 h per 1 SD (+0.0149 h per km, ≈ +1.5 h per 100 km).**
   The single largest driver. Network design, not execution: an order fulfilled from a
   forward/regional node 300 km closer arrives roughly **4.5 hours sooner**, every time.
   This is the quantitative case for micro-fulfilment and better warehouse-to-pincode
   sourcing rules.
2. **`Shipping_Mode` : Standard +5.00 h and Same_Day −2.62 h, both versus Express.**
   Holding distance and everything else constant, the service tier is worth about
   **7.6 hours** from Standard to Same_Day. This validates the price differential - and
   quantifies what a free-upgrade recovery actually buys a customer.
3. **`Warehouse_Processing_Hours` : +1.89 h per 1 SD (+1.01 h per hour).**
   A coefficient of ≈ 1.0 is the cleanest finding in the model: **every hour an order sits in
   the warehouse is an hour added to the customer's wait, one for one, with no absorption
   downstream.** Unlike distance or traffic, this is fully inside the team's control. Shaving
   90 minutes off average pick-pack-manifest time takes 90 minutes off every promise.

**Tier 2 - the network-condition drivers**

4. **`Traffic_Index` : +0.95 h per 1 SD (+0.53 h per point).** Moving a lane from moderate (5)
   to severe (9) congestion adds ≈ 2.1 hours. Congestion is not controllable, but it is
   *forecastable* - so it belongs in the promise shown at checkout, not in the post-hoc excuse.
5. **`Courier_Load_Index` : +0.60 h per 1 SD (+0.34 h per point).** Courier saturation costs
   about **20 minutes per index point**. A capacity-planning number: the delivery cost of
   under-staffing a route on a peak day is now explicit.

**Tier 3 - small but real**

6. **`Items_in_Order` : +0.24 h per item** and **`Package_Weight_Kg` : +0.04 h per kg** -
   genuine handling friction, but a 10-item order only adds ≈ 2.2 hours of handling versus a
   single-item order.
7. **`Product_Category`**: Electronics **+1.82 h** and Home_Kitchen **+1.24 h** versus Apparel
   (serialisation, fragile handling, bulk); Groceries **−0.96 h** (cold-chain lanes are
   deliberately fast). Books and Beauty are statistically indistinguishable from Apparel.

**What does *not* matter**

8. **`Order_Value` : ≈ +0.07 h per 1 SD, robust p ≈ 0.06 - economically zero.** High-value
   orders are *not* intrinsically slower once weight, item count and category are accounted
   for. Operationally useful: there is **no delivery-speed penalty for encouraging
   higher-value baskets**, and no case for a value-based routing rule.

**Intercept (15.76 h)** is the expected delivery time for an Apparel order shipped Express at
average distance, weight, dwell time, congestion and load - a sanity anchor, not a lever.

## B5. Residual analysis on the held-out test set

In [ ]:
pred_test = pipeline.predict(X_test)
res_test = y_test - pred_test

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].scatter(y_test, pred_test, s=14, alpha=.55)
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
ax[0].plot(lims, lims, "--", color="crimson")
ax[0].set_xlabel("Actual hours"); ax[0].set_ylabel("Predicted hours")
ax[0].set_title(f"Actual vs Predicted (R² = {r2_score(y_test, pred_test):.3f})")

ax[1].scatter(pred_test, res_test, s=14, alpha=.55); ax[1].axhline(0, color="crimson", ls="--")
ax[1].set_xlabel("Predicted hours"); ax[1].set_ylabel("Residual")
ax[1].set_title("Test residuals vs predicted")

sns.histplot(res_test, bins=30, kde=True, ax=ax[2])
ax[2].set_title(f"Test residuals (mean {res_test.mean():+.3f} h, sd {res_test.std():.3f} h)")
plt.tight_layout(); plt.show()

print("Share of test predictions within +/- 2 h of actual: "
      f"{(res_test.abs() <= 2).mean():.1%}")
print("Share within +/- 3 h: " f"{(res_test.abs() <= 3).mean():.1%}")

**Read.** Predictions track the 45° line across the entire 3-44 hour range with no visible
bias at either end; test residuals are centred on ~0 and roughly symmetric. About **82% of
test orders land within ±2 hours** of the predicted time and **94% within ±3 hours** - a
usable basis for a promised delivery window (quote the band, not the point estimate). The gentle widening of the residual spread at higher predicted values is the mild
heteroscedasticity documented in A9, and is why the app widens its band on long lanes.

## B6. Optional benchmark - Random Forest

Linear regression is the compulsory primary model. A Random Forest is fitted purely as a
check on whether meaningful non-linearity was left on the table.

In [ ]:
rf = Pipeline([("preprocessor", preprocessor),
               ("model", RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE,
                                               min_samples_leaf=2, n_jobs=-1))])
rf.fit(X_train, y_train)

bench = pd.DataFrame([
    {"model": "Linear Regression", **{k: v for k, v in evaluate(pipeline, X_test, y_test, "Test").items()
                                      if k in ("R2", "MAE", "RMSE")}},
    {"model": "Random Forest",     **{k: v for k, v in evaluate(rf, X_test, y_test, "Test").items()
                                      if k in ("R2", "MAE", "RMSE")}},
])
print(bench.round(4).to_string(index=False))

**Conclusion.** The Random Forest does **not** beat the linear model (slightly lower R² and
higher MAE/RMSE). That is the expected result once the assumption work is done: the
underlying process really is additive and linear, so the flexible model spends its capacity
fitting noise. Linear regression is therefore kept as the production model - it matches the
best available accuracy **and** hands operations a coefficient table it can act on.

---
# Part C - Save the Trained Pipeline
---
The full `ColumnTransformer` + `LinearRegression` pipeline is exported as a single artefact,
so the app performs no preprocessing of its own. The feature contract is exported alongside
it so the app can build its input widgets from the same source of truth.

In [ ]:
MODEL_PATH = "model.pkl"
joblib.dump(pipeline, MODEL_PATH)

# Feature contract + slider ranges for the app (kept next to the model, not hard-coded)
contract = {
    "target": TARGET,
    "target_unit": "hours",
    "numeric_features": NUM_FEATURES,
    "categorical_features": CAT_FEATURES,
    "feature_order": FEATURES,
    "categories": {c: sorted(df_model[c].unique().tolist()) for c in CAT_FEATURES},
    "numeric_ranges": {c: {"min": float(df_model[c].min()),
                           "max": float(df_model[c].max()),
                           "median": float(df_model[c].median())} for c in NUM_FEATURES},
    "train_metrics": {"r2_test": float(r2_score(y_test, pipeline.predict(X_test))),
                      "mae_test": float(mean_absolute_error(y_test, pipeline.predict(X_test))),
                      "rmse_test": float(np.sqrt(mean_squared_error(y_test, pipeline.predict(X_test))))},
}
joblib.dump(contract, "model_metadata.pkl")
df_model.to_csv("data_cleaned.csv", index=False)

print(f"Saved {MODEL_PATH}, model_metadata.pkl and data_cleaned.csv")

In [ ]:
# --- Reload sanity check: the app must get identical predictions -----------
loaded = joblib.load(MODEL_PATH)

sample = pd.DataFrame([{
    "Order_Value": 2500.0,
    "Package_Weight_Kg": 2.5,
    "Warehouse_Distance_Km": 250.0,
    "Items_in_Order": 3,
    "Warehouse_Processing_Hours": 4.0,
    "Courier_Load_Index": 6.0,
    "Traffic_Index": 6.5,
    "Product_Category": "Electronics",
    "Shipping_Mode": "Standard",
}])[FEATURES]

print("Sample prediction from reloaded pipeline: "
      f"{loaded.predict(sample)[0]:.2f} hours")
assert np.allclose(loaded.predict(X_test[:20]), pipeline.predict(X_test[:20]))
print("Reload check passed - saved pipeline reproduces in-notebook predictions exactly.")

---
## Summary

| Deliverable | Status |
|---|---|
| Data quality treated (missing, duplicates, impossible values, inconsistent categories) | Done - A1 |
| Feature roles defined, identifier excluded | Done - A2 |
| Linearity checked (scatter + partial-residual plots) | Satisfied, no transform needed - A3/A4 |
| Multicollinearity (correlation + VIF) | All VIF ≈ 1.0-1.3, no action - A5 |
| Independence of errors (Durbin-Watson) | DW ≈ 2.05, satisfied - A6 |
| Homoscedasticity (Breusch-Pagan + plots) | Mild violation, treated with HC3 robust inference - A7/A9 |
| Normality of residuals (histogram, Q-Q, JB) | Violated, fixed by influence treatment (JB p ≈ 0.87) - A8/A9 |
| Outliers / influence (std. residuals, leverage, Cook's D) | 6 exception rows removed with documented reasoning - A8b |
| Assumptions re-checked after correction | Done - A9 |
| 80/20 split, Pipeline, LinearRegression | Done - B1/B2 |
| R², Adj. R², MAE, MSE/RMSE, residual analysis | Test R² 0.968, MAE 1.22 h, RMSE 1.52 h - B3/B5 |
| Coefficients + business interpretation | Done - B4 |
| Optional second algorithm | Random Forest, does not beat linear - B6 |
| Model saved as `.pkl` | Done - Part C |

**Headline business finding:** delivery time is driven by *network design* (distance,
service tier) and *warehouse execution* (dwell time passes through hour-for-hour), not by
order economics. Order value has no effect on delivery speed at all.